# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlishaYaqub/FlyRank-ML-internship-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [41]:
%pip -q install duckdb
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

In [42]:
agg = con.sql("""
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS early_clicks,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS early_impressions,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS early_avg_position,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS late_clicks
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

agg["is_declining"] = (agg["late_clicks"] < agg["early_clicks"]).astype(int)

content_info = con.sql("""
    SELECT content_hash_id, content_updated_date, content_created_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""").df()
agg = agg.merge(content_info, on="content_hash_id", how="left")

cutoff = pd.Timestamp("2026-03-15")
updated = pd.to_datetime(agg["content_updated_date"])
created = pd.to_datetime(agg["content_created_date"])
agg["known_last_touch"] = updated.where(updated <= cutoff, created)
agg["days_since_update"] = (cutoff - agg["known_last_touch"]).dt.days
agg["days_since_update"] = agg["days_since_update"].clip(lower=0)

print(agg.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

['content_hash_id', 'early_clicks', 'early_impressions', 'early_avg_position', 'late_clicks', 'is_declining', 'content_updated_date', 'content_created_date', 'known_last_touch', 'days_since_update']


In [46]:
content_info = con.sql("""
    SELECT content_hash_id, content_updated_date, content_created_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""").df()

agg = agg.merge(content_info, on="content_hash_id", how="left")

In [47]:
cutoff = pd.Timestamp("2026-03-15")
updated = pd.to_datetime(agg["content_updated_date"])
created = pd.to_datetime(agg["content_created_date"])

agg["known_last_touch"] = updated.where(updated <= cutoff, created)
agg["days_since_update"] = (cutoff - agg["known_last_touch"]).dt.days
agg["days_since_update"] = agg["days_since_update"].clip(lower=0)

In [48]:
agg["staleness_bucket"] = pd.cut(
    agg["days_since_update"],
    bins=[-1, 60, 150, 300, 100000],
    labels=["fresh_0_60", "aging_60_150", "stale_150_300", "very_stale_300plus"]
)

staleness_table = agg.groupby("staleness_bucket")["is_declining"].agg(["mean", "count"])
print(staleness_table)

                        mean  count
staleness_bucket                   
fresh_0_60          0.159186  73568
aging_60_150        0.200885  25751
stale_150_300       0.155080  48549
very_stale_300plus  0.158504  28870


/tmp/ipykernel_5199/208409745.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_table = agg.groupby("staleness_bucket")["is_declining"].agg(["mean", "count"])


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1: staleness, linked to FlyRank's real refresh flag.
Claim: pages not updated in a long time are more likely to be declining.
Verdict: FALSE. Decline rate is nearly flat across all staleness buckets (0.155 to 0.201), with no clear upward trend as pages get older. Staleness alone does not predict decline in this data.

In [49]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
agg["staleness_bucket"] = pd.cut(
    agg["days_since_update"],
    bins=[-1, 60, 150, 300, 100000],
    labels=["fresh_0_60", "aging_60_150", "stale_150_300", "very_stale_300plus"]
)

staleness_table = agg.groupby("staleness_bucket")["is_declining"].agg(["mean", "count"])
print(staleness_table)

                        mean  count
staleness_bucket                   
fresh_0_60          0.159186  73568
aging_60_150        0.200885  25751
stale_150_300       0.155080  48549
very_stale_300plus  0.158504  28870


/tmp/ipykernel_5199/1305843050.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_table = agg.groupby("staleness_bucket")["is_declining"].agg(["mean", "count"])


Signal 2: CTR vs position mismatch, linked to FlyRank's real CTR-fix logic.
Claim: a page with a good position but low CTR for that position is a fixable problem, more likely to be declining.
Verdict: OPPOSITE. Pages underperforming their tier's average CTR actually decline less often (11.2%) than pages at or above average CTR (29.9%). This is likely because currently declining pages have very low volume, making their CTR noisy and sometimes coincidentally high, rather than genuinely healthy.

In [50]:
agg["ctr"] = agg["early_clicks"] / agg["early_impressions"].replace(0, pd.NA)

agg["position_bucket"] = pd.cut(
    agg["early_avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["top_3", "page_1", "page_2", "deep"]
)

ctr_table = agg.groupby("position_bucket")["ctr"].agg(["mean", "count"])
print(ctr_table)

                     mean  count
position_bucket                 
top_3            0.009315  16251
page_1           0.004655  70001
page_2           0.003337  26912
deep             0.002096  37511


/tmp/ipykernel_5199/1914363112.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_table = agg.groupby("position_bucket")["ctr"].agg(["mean", "count"])


In [51]:
agg["tier_avg_ctr"] = agg.groupby("position_bucket")["ctr"].transform("mean")
agg["ctr_underperforming"] = agg["ctr"] < agg["tier_avg_ctr"]

mismatch_table = agg.groupby("ctr_underperforming")["is_declining"].agg(["mean", "count"])
print(mismatch_table)

                         mean   count
ctr_underperforming                  
False                0.298791   49292
True                 0.111898  127446


/tmp/ipykernel_5199/494866225.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg["tier_avg_ctr"] = agg.groupby("position_bucket")["ctr"].transform("mean")


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

My rule: prioritize a page if it has meaningful search traffic (impressions) AND its CTR is at or above its position tier's average, since Signal 2 showed this group actually declines more often (29.9% vs 11.2%), the opposite of the common assumption. Staleness was dropped from the rule since Signal 1 showed no relationship.

Reason codes: HIGH_VALUE_AT_RISK (high volume, at/above-tier CTR), LOW_VOLUME_SKIP (too little traffic to prioritize).

Action labels: review_now (top scoring pages), monitor (mid scoring), no_action (low scoring or low volume).

In [52]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Score: prioritize by traffic volume, but only within the group our data showed actually declines more
agg["reason_code"] = agg["ctr_underperforming"].map({False: "HIGH_VALUE_AT_RISK", True: "LOW_VOLUME_SKIP"})
agg["score"] = agg["early_impressions"].where(~agg["ctr_underperforming"], 0)

agg["action"] = pd.cut(
    agg["score"].rank(pct=True),
    bins=[0, 0.5, 0.9, 1.0],
    labels=["no_action", "monitor", "review_now"]
)

queue = agg.sort_values("score", ascending=False)[
    ["content_hash_id", "score", "reason_code", "action", "early_impressions", "ctr", "is_declining"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(queue.head(10))

                 content_hash_id     score         reason_code      action  \
112636  content_eadb33b5df496f4a  161575.0  HIGH_VALUE_AT_RISK  review_now   
94213   content_e7b5dd4dff461ad2   87662.0  HIGH_VALUE_AT_RISK  review_now   
130956  content_f107e54b10b43725   74133.0  HIGH_VALUE_AT_RISK  review_now   
35162   content_512dbad65bd5ade9   67177.0  HIGH_VALUE_AT_RISK  review_now   
40814   content_0ec90963d98b97a5   67032.0  HIGH_VALUE_AT_RISK  review_now   
26723   content_9fff53e827550f9d   56470.0  HIGH_VALUE_AT_RISK  review_now   
129058  content_5fa2737c68998c2e   51874.0  HIGH_VALUE_AT_RISK  review_now   
122538  content_b2cb08ff59fcce78   51384.0  HIGH_VALUE_AT_RISK  review_now   
91382   content_b17c1d1cb0a346d6   51109.0  HIGH_VALUE_AT_RISK  review_now   
130570  content_605c93bd9a16c5a2   49754.0  HIGH_VALUE_AT_RISK  review_now   

        early_impressions       ctr  is_declining  
112636           161575.0  0.014823             0  
94213             87662.0  0.011042  

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top 10 review. My rule flags pages with high impressions and CTR at or above their position tier's average, since Signal 2 showed this group actually declines more often.

1. content_eadb33b5df496f4a, score 161575: flagged for its very high traffic and above-average CTR. Currently not declining. Would be wrong if this page's traffic keeps holding steady with no drop for months, meaning the CTR pattern from Signal 2 doesn't apply here.
2. content_e7b5dd4dff461ad2, score 87662: same reasoning, high volume and CTR. Currently not declining. Would be wrong if it never enters a decline phase despite the pattern.
3. content_f107e54b10b43725, score 74133: same reasoning. Not currently declining. Would be wrong if it stays stable long term.
4. content_512dbad65bd5ade9, score 67177: same reasoning, highest CTR in this group (1.7%). Not currently declining. Would be wrong if strong CTR here genuinely means healthy, not at risk.
5. content_0ec90963d98b97a5, score 67032: same reasoning. This one IS currently declining, supporting the rule.
6. content_9fff53e827550f9d, score 56470: same reasoning. Currently declining, supports the rule, though CTR here (0.49%) is close to average, a borderline case.
7. content_5fa2737c68998c2e, score 51874: same reasoning. Not currently declining. Would be wrong if this pattern never triggers for it.
8. content_b2cb08ff59fcce78, score 51384: same reasoning. Not currently declining.
9. content_b17c1d1cb0a346d6, score 51109: same reasoning. Currently declining, supports the rule.
10. content_605c93bd9a16c5a2, score 49754: same reasoning. Currently declining, supports the rule.

Overall: 4 of 10 top-ranked pages are currently declining, 6 are not yet. The rule is meant to catch risk before decline happens, so this is a reasonable, if imperfect, hit rate for a simple baseline.

In [53]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(queue.head(10)[["content_hash_id", "score", "reason_code", "is_declining"]])

                 content_hash_id     score         reason_code  is_declining
112636  content_eadb33b5df496f4a  161575.0  HIGH_VALUE_AT_RISK             0
94213   content_e7b5dd4dff461ad2   87662.0  HIGH_VALUE_AT_RISK             0
130956  content_f107e54b10b43725   74133.0  HIGH_VALUE_AT_RISK             0
35162   content_512dbad65bd5ade9   67177.0  HIGH_VALUE_AT_RISK             0
40814   content_0ec90963d98b97a5   67032.0  HIGH_VALUE_AT_RISK             1
26723   content_9fff53e827550f9d   56470.0  HIGH_VALUE_AT_RISK             1
129058  content_5fa2737c68998c2e   51874.0  HIGH_VALUE_AT_RISK             0
122538  content_b2cb08ff59fcce78   51384.0  HIGH_VALUE_AT_RISK             0
91382   content_b17c1d1cb0a346d6   51109.0  HIGH_VALUE_AT_RISK             1
130570  content_605c93bd9a16c5a2   49754.0  HIGH_VALUE_AT_RISK             1


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: rows 1, 2, 3, 7, and 8 are all not currently declining, meaning the rule flagged them purely on the assumption that high volume plus decent CTR predicts future risk, based on Signal 2's pattern. This assumption comes from a single month's snapshot, so these could easily be false positives if the pattern does not hold going forward. Row 4 has the highest CTR of the group (1.7 percent), which makes it the shakiest pick, since strong CTR usually reads as healthy, not at risk.

Leakage check: the score uses only early_impressions and ctr, both computed strictly from the first half of March (before the March 15 cutoff). late_clicks, the column used to build the is_declining label, was never used as a feature. No product flags such as health_score or quick-win tags were used anywhere in the rule. All reasoning is based on data knowable before the decision point.

In [54]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm the score and reason_code columns never touch label-derived or future-window fields
used_in_score = {"early_impressions", "ctr", "ctr_underperforming"}
forbidden = {"late_clicks", "is_declining", "health_score", "quick_win_flag"}
print("Any forbidden columns used in scoring:", used_in_score & forbidden)

Any forbidden columns used in scoring: set()


## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.